# 01 — Write Iceberg Tables with Polars

Demonstrates writing a Polars DataFrame into an Apache Iceberg table stored in RustFS (S3),
with the table registered in the Nessie REST catalog.

**Data flow:** `Polars DataFrame` → `PyArrow Table` → `PyIceberg append()` → `RustFS (Parquet files)` + `Nessie (metadata)`

**Prerequisites:** Run `00_setup_catalog.ipynb` first.

In [1]:
import datetime
import os

import polars as pl
from pyiceberg.catalog.rest import RestCatalog
from pyiceberg.schema import Schema
from pyiceberg.types import (
    DoubleType,
    LongType,
    NestedField,
    StringType,
    TimestampType,
)

NESSIE_URI       = os.environ["NESSIE_URI"]
S3_ENDPOINT      = os.environ["AWS_S3_ENDPOINT"]
S3_ACCESS_KEY    = os.environ["AWS_ACCESS_KEY_ID"]
S3_SECRET_KEY    = os.environ["AWS_SECRET_ACCESS_KEY"]
WAREHOUSE_BUCKET = os.environ["ICEBERG_WAREHOUSE_BUCKET"]
WAREHOUSE_URI    = f"s3://{WAREHOUSE_BUCKET}/warehouse"

catalog = RestCatalog(
    name="nessie",
    **{
        "uri": NESSIE_URI,
        "warehouse": WAREHOUSE_URI,
        "s3.endpoint": S3_ENDPOINT,
        "s3.access-key-id": S3_ACCESS_KEY,
        "s3.secret-access-key": S3_SECRET_KEY,
        "s3.path-style-access": "true",
        "s3.region": "us-east-1",
    },
)

print("Connected to catalog:", catalog.name)

Connected to catalog: nessie


## 1. Create a Polars DataFrame

In [2]:
df = pl.DataFrame(
    {
        "event_id":   ["e001", "e002", "e003", "e004", "e005"],
        "user_id":    [101, 102, 103, 101, 104],
        "event_type": ["click", "view", "click", "purchase", "view"],
        "amount":     [0.0, 0.0, 0.0, 49.99, 0.0],
        "ts": [
            datetime.datetime(2024, 1, 15, 10, 0, 0),
            datetime.datetime(2024, 1, 15, 11, 0, 0),
            datetime.datetime(2024, 1, 16,  9, 0, 0),
            datetime.datetime(2024, 1, 16, 14, 0, 0),
            datetime.datetime(2024, 1, 17,  8, 0, 0),
        ],
    }
)

print(df)
print("\nPolars schema:", df.schema)

shape: (5, 5)
┌──────────┬─────────┬────────────┬────────┬─────────────────────┐
│ event_id ┆ user_id ┆ event_type ┆ amount ┆ ts                  │
│ ---      ┆ ---     ┆ ---        ┆ ---    ┆ ---                 │
│ str      ┆ i64     ┆ str        ┆ f64    ┆ datetime[μs]        │
╞══════════╪═════════╪════════════╪════════╪═════════════════════╡
│ e001     ┆ 101     ┆ click      ┆ 0.0    ┆ 2024-01-15 10:00:00 │
│ e002     ┆ 102     ┆ view       ┆ 0.0    ┆ 2024-01-15 11:00:00 │
│ e003     ┆ 103     ┆ click      ┆ 0.0    ┆ 2024-01-16 09:00:00 │
│ e004     ┆ 101     ┆ purchase   ┆ 49.99  ┆ 2024-01-16 14:00:00 │
│ e005     ┆ 104     ┆ view       ┆ 0.0    ┆ 2024-01-17 08:00:00 │
└──────────┴─────────┴────────────┴────────┴─────────────────────┘

Polars schema: Schema([('event_id', String), ('user_id', Int64), ('event_type', String), ('amount', Float64), ('ts', Datetime(time_unit='us', time_zone=None))])


## 2. Define Iceberg schema and create table

In [3]:
iceberg_schema = Schema(
    NestedField(1, "event_id",   StringType(),    required=False),
    NestedField(2, "user_id",    LongType(),      required=False),
    NestedField(3, "event_type", StringType(),    required=False),
    NestedField(4, "amount",     DoubleType(),    required=False),
    NestedField(5, "ts",         TimestampType(), required=False),
)

TABLE_ID = ("demo", "events")

try:
    table = catalog.create_table(identifier=TABLE_ID, schema=iceberg_schema)
    print("Table created:", table.name())
except Exception as e:
    if "already exists" in str(e).lower():
        table = catalog.load_table(TABLE_ID)
        print("Table already exists — loaded:", table.name())
    else:
        raise

print("Location:", table.location())

Table created: ('nessie', 'demo', 'events')
Location: s3://iceberg-warehouse/warehouse/demo/events_417c2ce6-810f-498f-b449-7584bb88b8bb


/opt/conda/lib/python3.11/site-packages/pyiceberg/utils/deprecated.py:54: DeprecationWarning: Deprecated in 0.8.0, will be removed in 0.9.0. Table.identifier property is deprecated. Please use Table.name() function instead.
  _deprecation_warning(deprecation_notice(deprecated_in, removed_in, help_message))


## 3. Write Polars DataFrame to Iceberg

Polars → PyArrow → PyIceberg `append()`

In [4]:
arrow_table = df.to_arrow()

table.append(arrow_table)

print("✔ Data written.")
print("Snapshot ID:", table.current_snapshot().snapshot_id)
print("Files written:", len(list(table.scan().plan_files())))

/opt/conda/lib/python3.11/site-packages/pyiceberg/utils/deprecated.py:54: DeprecationWarning: Deprecated in 0.8.0, will be removed in 0.9.0. Table.identifier property is deprecated. Please use Table.name() function instead.
  _deprecation_warning(deprecation_notice(deprecated_in, removed_in, help_message))
/opt/conda/lib/python3.11/site-packages/pyiceberg/utils/deprecated.py:54: DeprecationWarning: Deprecated in 0.8.0, will be removed in 0.9.0. Support for parsing catalog level identifier in Catalog identifiers is deprecated. Please refer to the table using only its namespace and its table name.
  _deprecation_warning(deprecation_notice(deprecated_in, removed_in, help_message))


✔ Data written.
Snapshot ID: 1845556611847477001
Files written: 1


## 4. Verify — scan back as Polars

In [7]:
result = pl.from_arrow(table.scan().to_arrow())

print(f"Row count: {len(result)}")
result

Row count: 5


event_id,user_id,event_type,amount,ts
str,i64,str,f64,datetime[μs]
"""e001""",101,"""click""",0.0,2024-01-15 10:00:00
"""e002""",102,"""view""",0.0,2024-01-15 11:00:00
"""e003""",103,"""click""",0.0,2024-01-16 09:00:00
"""e004""",101,"""purchase""",49.99,2024-01-16 14:00:00
"""e005""",104,"""view""",0.0,2024-01-17 08:00:00


## 5. Append more data (creates a new snapshot)

Each `append()` creates a new immutable snapshot — the basis for time travel.

In [8]:
df2 = pl.DataFrame(
    {
        "event_id":   ["e006", "e007"],
        "user_id":    [105, 101],
        "event_type": ["click", "purchase"],
        "amount":     [0.0, 99.00],
        "ts": [
            datetime.datetime(2024, 1, 18, 9, 0, 0),
            datetime.datetime(2024, 1, 18, 10, 30, 0),
        ],
    }
)

table.append(df2.to_arrow())

print("✔ Second batch written.")
print("Snapshots:", len(table.history()))
print("Total rows:", len(pl.from_arrow(table.scan().to_arrow())))

✔ Second batch written.
Snapshots: 1
Total rows: 7


/opt/conda/lib/python3.11/site-packages/pyiceberg/utils/deprecated.py:54: DeprecationWarning: Deprecated in 0.8.0, will be removed in 0.9.0. Table.identifier property is deprecated. Please use Table.name() function instead.
  _deprecation_warning(deprecation_notice(deprecated_in, removed_in, help_message))
/opt/conda/lib/python3.11/site-packages/pyiceberg/utils/deprecated.py:54: DeprecationWarning: Deprecated in 0.8.0, will be removed in 0.9.0. Support for parsing catalog level identifier in Catalog identifiers is deprecated. Please refer to the table using only its namespace and its table name.
  _deprecation_warning(deprecation_notice(deprecated_in, removed_in, help_message))
